In [184]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import category_encoders as ce

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error

from sklearn.model_selection import GridSearchCV
from skopt import BayesSearchCV

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv('/home/puru/Documents/House Price Prediction/STEP - 3 (Feature Engg)/gurgaon_properties_post_feature_selection.csv')

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,super_built_up_area,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 47,2.40,3,3,0,Moderately Old,2992.86,No,semifurnished,Low,Low Floor
1,flat,sector 65,2.36,3,3,3,Relatively New,1650.00,Yes,semifurnished,Medium,Low Floor
2,flat,sector 111,2.25,3,3,2,Relatively New,1700.05,No,unfurnished,Medium,High Floor
3,flat,sector 71,0.60,2,2,1,Relatively New,599.98,No,unfurnished,Medium,Mid Floor
4,flat,sector 2,1.40,3,3,3+,Moderately Old,1780.03,No,unfurnished,Medium,Mid Floor


In [4]:
X = df.drop(columns=['price'])
y = df['price']

In [5]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,super_built_up_area,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 47,3,3,0,Moderately Old,2992.86,No,semifurnished,Low,Low Floor
1,flat,sector 65,3,3,3,Relatively New,1650.00,Yes,semifurnished,Medium,Low Floor
2,flat,sector 111,3,3,2,Relatively New,1700.05,No,unfurnished,Medium,High Floor
3,flat,sector 71,2,2,1,Relatively New,599.98,No,unfurnished,Medium,Mid Floor
4,flat,sector 2,3,3,3+,Moderately Old,1780.03,No,unfurnished,Medium,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...
3506,flat,sector 85,2,2,3,Relatively New,1670.99,No,unfurnished,Low,High Floor
3507,flat,sector 49,3,3,3,Relatively New,2032.98,No,unfurnished,Low,High Floor
3508,flat,sector 85,2,2,3,Relatively New,1639.99,No,unfurnished,Low,Mid Floor
3509,flat,sector 37d,4,4,3,Relatively New,2190.99,No,semifurnished,Medium,High Floor


In [6]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

# `1.`Ordinal Encoding

In [7]:
columns_to_encode = ['property_type','sector','balcony','agePossession','servant room','furnishing_type','luxury_category','floor_category']

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['super_built_up_area','bedRoom','bathroom']),
        ('cat', OrdinalEncoder(handle_unknown='error' ), columns_to_encode)], 
        remainder='passthrough')

In [9]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
    ])

In [10]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [11]:
scores.mean(),scores.std()

(0.7236284009897963, 0.01888812009015952)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [13]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['super_built_up_area',
                                                   'bedRoom', 'bathroom']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'servant room',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [14]:
y_pred = pipeline.predict(X_test)

In [15]:
y_pred = np.expm1(y_pred)

In [16]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.8490403556326911

In [17]:
def scorer(model_name, model):
    output = []
    output.append(model_name)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)])
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',n_jobs=-1)
    output.append(scores.mean())
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    return output
    

In [18]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [19]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [20]:
model_output

[['linear_reg', 0.7236284009897963, 0.8490403556326911],
 ['svr', 0.741051853167481, 0.8155992365610815],
 ['ridge', 0.7236336012579859, 0.8489093607026024],
 ['LASSO', 0.052202509886864826, 1.4070462694408605],
 ['decision tree', 0.7707562009226894, 0.6251327060619972],
 ['random forest', 0.8813771674690271, 0.4661354683085518],
 ['gradient boosting', 0.8744225327663886, 0.4993680308805494],
 ['adaboost', 0.7406810200078159, 0.7309419922010111],
 ['mlp', 0.8150574203006068, 0.6528162487582011],
 ['xgboost', 0.8877244671117408, 0.47901066519966506]]

In [21]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [24]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.881377,0.466135
9,xgboost,0.887724,0.479011
6,gradient boosting,0.874423,0.499368
4,decision tree,0.770756,0.625133
8,mlp,0.815057,0.652816
7,adaboost,0.740681,0.730942
1,svr,0.741052,0.815599
2,ridge,0.723634,0.848909
0,linear_reg,0.723628,0.849040
3,LASSO,0.052203,1.407046


# `2.` OneHotEncoding

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['super_built_up_area','bedRoom','bathroom']),
        ('cat',OneHotEncoder(drop='first'),columns_to_encode)], 
        remainder='passthrough')

In [23]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [25]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [26]:
scores.mean()

0.8561165653650814

In [27]:
scores.std()

0.009895154925747314

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [29]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['super_built_up_area',
                                                   'bedRoom', 'bathroom']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'servant room',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [30]:
y_pred = pipeline.predict(X_test)

In [31]:
y_pred = np.expm1(y_pred)

In [32]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.5863941090257259

In [33]:
def scorer(model_name, model):
    output = []
    output.append(model_name)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)])
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',n_jobs=-1)
    output.append(scores.mean())
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    return output

In [34]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [35]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [36]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [37]:
model_df.sort_values(['mae'])

,name,r2,mae
9,xgboost,0.891352,0.445119
1,svr,0.895187,0.462562
5,random forest,0.869293,0.465522
8,mlp,0.885434,0.475946
6,gradient boosting,0.853512,0.538739
0,linear_reg,0.856117,0.586394
2,ridge,0.856343,0.591869
4,decision tree,0.785481,0.593579
7,adaboost,0.713543,0.808105
3,LASSO,-0.004740,1.440246


### `3.` Target Encoder

In [66]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['super_built_up_area']),
        ('cat', OrdinalEncoder(), ['balcony','bedRoom', 'bathroom']),
        ('cat1',OneHotEncoder(drop='first'),['property_type','agePossession','servant room','furnishing_type','luxury_category','floor_category']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ]
)

In [67]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR())
])

In [68]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',n_jobs=-1)

In [69]:
scores.mean(),scores.std()

(0.8639729470317171, 0.013352561872698666)

In [70]:
def scorer(model_name, model):
    output = []
    output.append(model_name)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)])
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',n_jobs=-1)
    output.append(scores.mean())
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    return output

In [71]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [72]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [73]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [74]:
model_df.sort_values(['mae'])

,name,r2,mae
9,xgboost,0.894617,0.445895
5,random forest,0.895587,0.463334
6,gradient boosting,0.881795,0.488411
1,svr,0.863973,0.545905
4,decision tree,0.821406,0.558189
8,mlp,0.851896,0.580777
7,adaboost,0.809230,0.639755
0,linear_reg,0.823710,0.677131
2,ridge,0.823736,0.677385
3,LASSO,-0.004740,1.440246


# `Hyperparameter Tuning`

## We got the best result we `xgboost` so we tune it.